In [9]:
import os
# os.environ['HTA_DISABLE_NS_ROUNDING'] = '1'
os.environ['CRITICAL_PATH_ADD_ZERO_WEIGHT_LAUNCH_EDGE'] = '1'
from hta.trace_analysis import TraceAnalysis
base_dir = "/mnt/self-define/zhangweixing/profiler_log/deepseek_v2/tensorboard/"
# trace_dir = base_dir + "20250325-1043_pretrain-zjmcore-dsv3-16B-lr-4E-5-minlr-4E-6-bs-1-gbs-32-seqlen-4096-pr-bf16-pp-2-ac-none/"
trace_dir = base_dir + "z2000-16b/"

analyzer = TraceAnalysis(trace_dir=trace_dir)

In [ ]:
instance_id = 0  # note this is zero based
annotation = "ProfilerStep"#"ProfilerStep"  # will match multiple ProfilerStepXXX annotations
cp_graph, success = analyzer.critical_path_analysis(rank = 0, annotation=annotation, instance_id=instance_id)

In [11]:
analyzer.overlay_critical_path_analysis(rank=0, critical_path_graph = cp_graph, output_dir=trace_dir + "/overlay", only_show_critical_events=True)

'/mnt/self-define/zhangweixing/profiler_log/deepseek_v2/tensorboard/z2000-16b/overlay/overlaid_critical_path_dlcppn30ol2cuyku-master-0_105.1744099060184936479.pt.trace.json'

In [12]:
cp_graph.summary()

Critical Path broken down by boundedness = (in % of duration)


bound_by
                               0.000000
cpu_bound                     46.483775
gpu_communication_bound       33.125646
gpu_compute_bound              5.463277
gpu_kernel_kernel_overhead    12.667616
gpu_kernel_launch_overhead     2.259686
Name: duration, dtype: float64

In [13]:
cp_dp = cp_graph.get_critical_path_breakdown()
cp_dp

,event_idx,duration,type,s_name,cat,pid,tid,stream,index,bound_by
0,126641.0,48043.0,critical_path_operator,_GroupedLinear,556.0,105,105,-1.0,126641.0,cpu_bound
1,147932.0,6973.0,critical_path_operator,cuLaunchKernelEx,476.0,105,105,-1.0,147932.0,cpu_bound
2,76992.0,137.0,critical_path_operator,aten::narrow,556.0,105,1294,-1.0,76992.0,cpu_bound
3,10753.0,122.0,critical_path_operator,torch::autograd::AccumulateGrad,556.0,105,1294,-1.0,10753.0,cpu_bound
4,155818.0,1052.0,critical_path_operator,cudaStreamWaitEvent,223.0,105,105,-1.0,155818.0,cpu_bound
...,...,...,...,...,...,...,...,...,...,...
222401,241896.0,1696.0,critical_path_kernel_kernel_delay,transformer_engine::normalization::rmsnorm_fwd...,249.0,0,7,7.0,241896.0,gpu_kernel_kernel_overhead
222402,28475.0,344.0,critical_path_operator,aten::split_with_sizes,556.0,105,1294,-1.0,28475.0,cpu_bound
222403,147314.0,4367.0,critical_path_operator,cudaLaunchKernel,223.0,105,105,-1.0,147314.0,cpu_bound
222404,164392.0,2581.0,critical_path_operator,cudaMemsetAsync,223.0,105,105,-1.0,164392.0,cpu_bound


In [14]:
import networkx as nx
import matplotlib.pyplot as plt

print(nx.is_weakly_connected(cp_graph))
print(cp_graph.number_of_nodes())
print(cp_graph.number_of_edges())
print("Connected components:", len(list(nx.weakly_connected_components(cp_graph))))
# for component in nx.weakly_connected_components(cp_graph):
#     subgraph = cp_graph.subgraph(component)
#     print(subgraph.number_of_nodes())
#     print(subgraph.number_of_edges())
#     print(nx.is_weakly_connected(subgraph))
    


# # 绘制图
# pos = nx.random_layout(cp_graph)  # 使用 spring 布局算法来定位节点
# nx.draw(cp_graph, pos, with_labels=True, node_color='skyblue', node_size=800, edge_color='gray', linewidths=2, font_size=15)
 
# # 显示图形
# plt.show()

False
399334
417982
Connected components: 3


In [15]:
print(cp_dp['duration'].sum())

2748262385.0


In [19]:
import pandas as pd
PROFILE_TIMES={}
rank=0
t = analyzer.t
t.decode_symbol_ids()
trace_df: pd.DataFrame = t.get_trace(rank)
sym_index = t.symbol_table.get_sym_id_map()
annotation = "ProfilerStep"  # will match multiple ProfilerStepXXX annotations
instance_id = 0  # note this is zero based
annotation_ids = [val for key, val in sym_index.items() if annotation in key]
instance_start, instance_end = instance_id, instance_id
annotations = trace_df[trace_df.name.isin(annotation_ids)].copy()
annotations["end_ts"] = annotations["ts"] + annotations["dur"]

start_ts = annotations.ts[instance_start : instance_end + 1].min()
end_ts = annotations.end_ts[instance_start : instance_end + 1].max()


In [21]:
cpu_kernels = trace_df[trace_df["stream"].eq(-1)]
stream_wait_event_id = sym_index.get("Stream Wait Event", -200)
a = cpu_kernels.query(f"(ts >= {start_ts} and ts <= {end_ts}) and (dur > 0)")
gpu_kernels = trace_df[trace_df["stream"].ne(-1)]
cpu_kernels = cpu_kernels.copy().set_index("index_correlation")
b = (
    gpu_kernels[["ts", "dur", "correlation", "name"]]
    .join(cpu_kernels[["ts", "dur"]], rsuffix="_runtime")
    .query(
        f"(ts_runtime >= {start_ts} and ts_runtime <= {end_ts} and dur_runtime > 0)"
        f" or (name == {stream_wait_event_id})"
    )
)

clipped_df = trace_df.loc[a.index.union(b.index)].copy()

In [30]:
from copy import deepcopy
t_copy = deepcopy(t)
t_copy.traces[rank] = clipped_df

In [31]:
t_full = t
t = t_copy

In [32]:
full_trace_df: pd.DataFrame = t_full.get_trace(rank)
sym_table = t_full.symbol_table.get_sym_table()
symbol_table = t_full.symbol_table

In [33]:
trace_df: pd.DataFrame = t.get_trace(rank)

In [36]:
events_df = (
            trace_df.query(
                symbol_table.get_operator_or_cuda_runtime_query()
                + " or (stream != -1 and index_correlation >= 0)"
            )[["index", "ts", "dur", "name"]]
        ).rename(columns={"index": "ev_idx"})

In [59]:
trace_df[trace_df["pid"]==0].groupby(['stream','s_cat', 'stream']).size()

stream  s_cat                stream
-1      cuda_sync            -1         2036
        gpu_user_annotation  -1         1697
 7      cuda_sync             7         2054
        gpu_memcpy            7          735
        gpu_memset            7          859
        kernel                7        16814
 22     cuda_sync             22         534
        kernel                22         267
 26     cuda_sync             26          84
        kernel                26          42
 30     cuda_sync             30          78
        kernel                30          39
 34     cuda_sync             34           2
        kernel                34           1
 38     cuda_sync             38           6
        kernel                38           3
 42     cuda_sync             42           2
        kernel                42           1
 50     cuda_sync             50           2
        kernel                50           1
 142    cuda_sync             142        216
        kernel     

In [51]:
with pd.option_context("display.max_columns", None, "display.max_rows", None):
    display(
trace_df.groupby(['pid', 's_cat']).size())

pid  s_cat              
0    cuda_sync                6478
     gpu_memcpy                735
     gpu_memset                972
     gpu_user_annotation      1697
     kernel                  22386
105  cpu_op                 124883
     cuda_driver              6838
     cuda_runtime            43853
     user_annotation          2879
dtype: int64

In [60]:
trace_df

,index,cat,name,pid,tid,ts,dur,stream,correlation,bytes,...,wait_on_cuda_event_record_corr_id,input_dims,input_type,input_strides,external_id,index_correlation,iteration,end,s_name,s_cat
0,0,292,251,105,1294,7.347809e+05,186.654,-1,-1,-1,...,-1,-1,-1,-1,22529,-1,10,5.774581e+11,autograd::engine::evaluate_function: CompiledF...,cpu_op
1,1,292,534,105,1294,7.347917e+05,160.442,-1,-1,-1,...,-1,-1,-1,-1,22530,-1,10,5.774581e+11,CompiledFunctionBackward,cpu_op
3,3,292,95,105,1294,7.349833e+05,10.958,-1,-1,-1,...,-1,-1,-1,-1,22531,-1,10,5.774581e+11,autograd::engine::evaluate_function: AddBackward0,cpu_op
4,4,292,463,105,1294,7.349854e+05,2.522,-1,-1,-1,...,-1,-1,-1,-1,22532,-1,10,5.774581e+11,AddBackward0,cpu_op
6,6,292,59,105,1294,7.350018e+05,494.143,-1,-1,-1,...,-1,-1,-1,-1,22533,-1,10,5.774581e+11,autograd::engine::evaluate_function: _LinearBa...,cpu_op
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
303742,303742,147,217,0,161,6.998429e+05,1199.458,-1,-1,-1,...,-1,-1,-1,-1,20903,-1,10,5.774581e+11,experts_te_grouped_mlp_fc2,gpu_user_annotation
303743,303743,147,217,0,161,1.013707e+06,1051.746,-1,-1,-1,...,-1,-1,-1,-1,30461,-1,10,5.774584e+11,experts_te_grouped_mlp_fc2,gpu_user_annotation
303744,303744,147,217,0,161,7.194995e+05,983.074,-1,-1,-1,...,-1,-1,-1,-1,21722,-1,10,5.774581e+11,experts_te_grouped_mlp_fc2,gpu_user_annotation
303745,303745,147,217,0,161,5.145019e+05,1175.073,-1,-1,-1,...,-1,-1,-1,-1,12713,-1,10,5.774579e+11,experts_te_grouped_mlp_fc2,gpu_user_annotation
